In [11]:
import pandas as pd
import numpy as np

from skeLCS.eLCS import eLCS

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

In [12]:
df = pd.read_csv("../data/weatherAUS_compact.csv")

print("Dataset shape:", df.shape)
print(df.head())
print("\nMissing values:", df.isna().sum().sum())
print("\nData types:")
print(df.dtypes.value_counts())

Dataset shape: (142193, 28)
   Location  MinTemp  MaxTemp  Rainfall  Evaporation  Sunshine  WindGustSpeed  \
0         2     13.4     22.9       0.6          4.8       8.4           44.0   
1         2      7.4     25.1       0.0          4.8       8.4           44.0   
2         2     12.9     25.7       0.0          4.8       8.4           46.0   
3         2      9.2     28.0       0.0          4.8       8.4           24.0   
4         2     17.5     32.3       1.0          4.8       8.4           41.0   

   WindSpeed9am  WindSpeed3pm  Humidity9am  ...  RainTomorrow  Year  Month  \
0          20.0          24.0         71.0  ...             0  2008     12   
1           4.0          22.0         44.0  ...             0  2008     12   
2          19.0          26.0         38.0  ...             0  2008     12   
3          11.0           9.0         45.0  ...             0  2008     12   
4           7.0          20.0         82.0  ...             0  2008     12   

   Day  WindGust

In [13]:
print(df["RainTomorrow"].value_counts())
print(df["RainTomorrow"].value_counts(normalize=True))

RainTomorrow
0    110316
1     31877
Name: count, dtype: int64
RainTomorrow
0    0.775819
1    0.224181
Name: proportion, dtype: float64


In [14]:
x = df.drop(columns=["RainTomorrow"])
y = df["RainTomorrow"]

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training size:", len(x_train))
print("Testing size:", len(x_test))
print("Features:", x_train.shape[1])

Training size: 113754
Testing size: 28439
Features: 27


In [15]:
discrete_cols = [
    "Location",
    "RainToday"
]

discrete_attributes = np.array([
    x_train.columns.get_loc(col)
    for col in discrete_cols
])

print("Discrete columns:", discrete_cols)
print("Discrete indices:", discrete_attributes)

for col in discrete_cols:
    print(col, "unique values:", x_train[col].nunique())

Discrete columns: ['Location', 'RainToday']
Discrete indices: [ 0 17]
Location unique values: 49
RainToday unique values: 2


In [16]:
x_train_np = x_train.to_numpy(dtype=float)
x_test_np = x_test.to_numpy(dtype=float)

y_train_np = y_train.to_numpy(dtype=float)
y_test_np = y_test.to_numpy(dtype=float)

print("x_train:", x_train_np.shape)
print("x_test:", x_test_np.shape)
print("y_train:", y_train_np.shape)
print("y_test:", y_test_np.shape)

x_train: (113754, 27)
x_test: (28439, 27)
y_train: (113754,)
y_test: (28439,)


In [17]:
from sklearn.model_selection import train_test_split

x_sample, _, y_sample, _ = train_test_split(
    x_train,
    y_train,
    train_size=10000,
    random_state=42,
    stratify=y_train
)

print("Sample shape:", x_sample.shape)
print("\nSample class distribution:")
print(y_sample.value_counts())

Sample shape: (10000, 27)

Sample class distribution:
RainTomorrow
0    7758
1    2242
Name: count, dtype: int64


In [18]:
from skeLCS import eLCS

model = eLCS(
    learning_iterations=10000,
    N=7000,
    random_state=42
)

_ = model.fit(
    x_sample.values,
    y_sample.values
)

print("Training complete")

Training complete


In [19]:
predictions = model.predict(x_test.values)

In [20]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

print("Accuracy:", accuracy_score(y_test, predictions))
print(
    "Balanced Accuracy:",
    balanced_accuracy_score(y_test, predictions)
)

print("\nClassification Report:")
print(classification_report(y_test, predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

Accuracy: 0.7984809592461057
Balanced Accuracy: 0.5566445065335779

Classification Report:
              precision    recall  f1-score   support

           0       0.80      1.00      0.88     22064
           1       0.87      0.12      0.21      6375

    accuracy                           0.80     28439
   macro avg       0.83      0.56      0.55     28439
weighted avg       0.81      0.80      0.73     28439


Confusion Matrix:
[[21954   110]
 [ 5621   754]]


In [21]:
clean_results = pd.DataFrame({
    "actual": y_test.to_numpy(),
    "prediction": predictions
})

clean_results.to_csv(
    "clean_elcs_predictions.csv",
    index=False
)

model.export_iteration_tracking_data(
    "clean_elcs_iterations.csv"
)

print("Clean predictions and iteration data saved")
clean_results.head()

Clean predictions and iteration data saved


,actual,prediction
0,0,0
1,1,0
2,0,0
3,0,0
4,0,0


In [23]:
headers = x_train.columns.values
classLabel = "RainTomorrow"

model.export_iteration_tracking_data(
    "clean_elcs_iterations.csv"
)

model.export_final_rule_population(
    headers,
    classLabel,
    filename="clean_elcs_rules.csv",
    DCAL=False
)

print("Iteration data and rule population saved")

Iteration data and rule population saved
